In [ ]:
import pandas as pd
import json
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Cargar configuración
with open("../config.json", "r") as f:
    config = json.load(f)

# Cargar datos
df = pd.read_csv(f"../{config['data']['path']}")

# Separar features y target
target = config["data"]["target_column"]
X = df.drop(columns=[target])
y = df[target]

# Identificar tipos de variables
categorical = X.select_dtypes(include="object").columns.tolist()
numerical = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

# Pipelines
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numerical),
    ("cat", categorical_pipeline, categorical)
])

# Aplicar transformaciones
X_transformed = preprocessor.fit_transform(X)

# Separar datos
X_train, X_test, y_train, y_test =  train_test_split(X_transformed, y, test_size=0.2, random_state=42)
